# SFT 1단계 — RFT 학습 데이터 만들기

## 이 노트북이 하는 일
베이스 모델로 train 문제를 여러 번 풀게 하고, **정답을 맞힌 풀이만 골라내서** 학습 데이터로 저장합니다.

```
train 문제  →  모델이 4번씩 풀이  →  정답 맞힌 것만 채택  →  train_rft.jsonl
```

## 왜 이걸 하나
진단 결과 `pass@32=0.8633` / `maj@32=0.7433`.
**정답 경로를 찾을 줄은 아는데 확률이 낮아서 다수결에서 지는 문제가 12%p** 있습니다.
정답 풀이로 학습하면 그 경로의 확률이 올라가서 `maj@k`가 `pass@k` 쪽으로 끌려 올라갑니다.

## 실행 순서
1. Settings: **Accelerator = GPU T4 x2**, **Internet = On**
2. `[1]` → `[2]` → `[3]` 실행
3. **Run → Restart Session**
4. `[1]` 다시 실행 → `[4]`부터 순서대로

## ⚠️ 이 노트북의 핵심 안전장치
로컬 검증셋 300문제(`seed=42`)를 **먼저 제외**합니다.
안 그러면 검증 문제로 학습하고 그 문제로 채점하게 되어 점수가 가짜로 부풀려집니다.


---
## [1] 설정 ▶️ 항상 실행

### 🤔 `N_PROBLEMS`를 왜 8000으로?
train 전체는 16,373문제입니다. 전부 쓰면 생성에만 **약 5.7시간**이 걸려요.
Kaggle 주간 GPU 할당량(약 30시간)을 8/31 최종 추론용으로 남겨야 하므로 절반만 씁니다.
데이터가 부족하면 나중에 늘리면 됩니다.

### 🤔 `TEMP=1.0`으로 높인 이유
추론할 때는 0.8이었지만, 여기서는 **정답 경로를 찾는 게 목적**입니다.
무작위성을 높이면 평소 안 가던 경로도 탐색해서, 어려운 문제에서 정답을 건질 확률이 올라갑니다.

### 🤔 `K=4`
문제당 4번 풀립니다. 늘리면 데이터가 많아지지만 시간이 비례해서 늘어납니다.

In [ ]:
# ── 설정 ────────────────────────────────────────────
N_PROBLEMS = 8000      # RFT에 사용할 train 문제 수 (최대 16373)
K          = 4         # 문제당 생성 횟수
TEMP       = 1.0       # 탐색을 위해 추론(0.8)보다 높게
MAX_TOKENS = 1024
VALID_N    = 300       # 검증셋 크기 — 추론 노트북과 반드시 동일
SEED       = 42        # 추론 노트북과 반드시 동일 (검증셋을 정확히 제외하기 위해)
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
OUT_PATH   = "/kaggle/working/train_rft.jsonl"
# ────────────────────────────────────────────────────
print(f"{N_PROBLEMS}문제 x {K}회 = {N_PROBLEMS*K:,}회 생성 예정")
print(f"예상 소요: 약 {N_PROBLEMS*K*0.315/3600:.1f}시간")

---
## [2] vLLM 설치 ⏭️ 세션을 끄지 않았으면 건너뛰기

In [ ]:
!pip install -q -U vllm 2>&1 | tail -3

---
## [3] protobuf 버전 고정 ⏭️ [2]와 함께 건너뛰기

Kaggle 기본 protobuf(5.x)와 vLLM 의존성(6.x) 충돌 해결. **`<7` 상한 필수.**

---
## ⛔ 여기서 Run → Restart Session
설치는 디스크에, 적용은 재시작에. 재시작 후 **[1]부터** 다시 실행.

---

In [ ]:
!pip install -q -U "protobuf>=6.33.6,<7" 2>&1 | tail -2
import google.protobuf as p
print("protobuf", p.__version__)
assert p.__version__.startswith("6."), "protobuf가 6.x가 아닙니다!"

---
## [4] 데이터 로드 + 검증셋 제외 ▶️

### 이 셀의 3단계
1. train 17,000 → 오류 627개 제거 → **16,373**
2. **검증셋 300문제 제외** → 16,073  ← 오염 방지, 가장 중요
3. 거기서 `N_PROBLEMS`개 샘플링

### 🤔 왜 `random_state=SEED`가 똑같아야 하나
추론 노트북에서 `train.sample(300, random_state=42)`로 검증셋을 뽑았습니다.
**같은 순서로 필터링하고 같은 시드를 써야** 정확히 같은 300문제가 재현됩니다.
하나라도 다르면 다른 300문제가 나와서 제외가 무의미해집니다.

In [ ]:
import glob, os, pandas as pd

def find_csv(must_have, must_not=()):
    for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        b = os.path.basename(p).lower()
        if all(k in b for k in must_have) and not any(k in b for k in must_not):
            return p
    return None

TRAIN_PATH = find_csv(["train"], must_not=["filtered", "ids", "leaderboard", "test"])
BAD_PATH   = find_csv(["filtered", "ids"])
assert TRAIN_PATH and TRAIN_PATH != BAD_PATH, "경로 탐색 실패"

train = pd.read_csv(TRAIN_PATH)
bad   = set(pd.read_csv(BAD_PATH)["id"])
train = train[~train["id"].isin(bad)].reset_index(drop=True)
print(f"오류 문항 제거 후: {len(train)}")
assert len(train) == 16373, f"16373이어야 하는데 {len(train)}입니다. 추론 노트북과 조건이 다릅니다!"

# ⚠️ 검증셋 제외 — 추론 노트북과 완전히 동일한 방식이어야 함
valid_ids = set(train.sample(VALID_N, random_state=SEED)["id"])
pool = train[~train["id"].isin(valid_ids)].reset_index(drop=True)
print(f"검증셋 {len(valid_ids)}문제 제외 후: {len(pool)}")
assert len(pool) == 16373 - VALID_N

work = pool.sample(min(N_PROBLEMS, len(pool)), random_state=SEED).reset_index(drop=True)
print(f"RFT 대상: {len(work)}문제")
work.head(2)

---
## [5] 답 추출기 ▶️
추론 노트북과 **완전히 동일**합니다. 정답 판정에 쓰입니다.

In [ ]:
import re
from collections import Counter

def extract_boxed(text):
    """Return the raw content inside the LAST \\boxed{...}, brace-balanced."""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # bare form: \boxed 15
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/text -> python int, or None. Never uses float(), so huge ints survive."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')          # LaTeX thousands separator
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)                       # trailing unit: 42cm -> 42
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':     # (\frac{100}{4}) -> \frac{100}{4}
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None means 'this sample produced no usable integer' -> dropped from voting."""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)          # boxed present but unparseable -> None, do NOT guess
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]

_cases = [(r"\boxed{132}", 132), (r"\boxed{-2,025,078}", -2025078),
          (r"\boxed{\dfrac{650}{5}}", 130), (r"\boxed{\frac{7}{2}}", None)]
print("parser FAILURES:", sum(parse_answer(t) != w for t, w in _cases), "/", len(_cases))

---
## [6] 프롬프트 구성 ▶️
추론 때와 **같은 시스템 프롬프트**를 씁니다.

### 🤔 왜 같아야 하나
학습 데이터의 형식과 실제 추론 형식이 다르면, 모델이 학습한 것과 다른 상황을 만나게 됩니다.
**학습과 추론의 조건을 일치시키는 것**이 SFT의 기본 원칙입니다.

In [ ]:
from transformers import AutoTokenizer

SYSTEM = ("You are an expert competition mathematician. Solve the problem step by step, "
          "concisely. The final answer is ALWAYS a single integer. "
          "End your response with the final integer inside \\boxed{}.")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
prompts = [
    tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
        tokenize=False, add_generation_prompt=True)
    for q in work["question"]
]
print(f"{len(prompts)}문제 x {K}샘플 = {len(prompts)*K:,}회 생성")

---
## [7] 생성 ▶️ 가장 오래 걸림 (약 3시간)

⚠️ **두 번 실행 금지.** GPU 메모리를 이미 잡고 있어 에러가 납니다.

세션 12시간 제한이 있으니, `N_PROBLEMS`가 크면 중간에 끊길 위험을 계산해 두세요.
8000문제 × 4샘플이면 약 3시간이라 안전합니다.

In [ ]:
import time
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_ID, dtype="half", max_model_len=4096,
          gpu_memory_utilization=0.90, tensor_parallel_size=1,
          seed=SEED, trust_remote_code=True)

sp = SamplingParams(n=K, temperature=TEMP, top_p=0.95,
                    max_tokens=MAX_TOKENS, seed=SEED)

t0 = time.time()
outs = llm.generate(prompts, sp)
print(f"\n생성 완료: {(time.time()-t0)/60:.1f}분")

---
## [8] 정답 필터링 + 난이도별 채택 ▶️ (GPU 미사용, 몇 초)

### 이 셀의 핵심 판단

**① 정답을 맞힌 풀이만 채택** — 이게 "Rejection Sampling"의 rejection입니다.

**② 난이도에 따라 채택 개수를 다르게**

| 4번 중 정답 | 의미 | 채택 |
|---|---|---|
| 4/4 | 이미 확실히 아는 문제 | **1개만** |
| **1~3/4** | **흔들리는 문제 = 우리 목표** | **최대 2개** |
| 0/4 | 못 푸는 문제 | 0개 (버림) |

4/4 문제를 다 넣으면 **이미 잘하는 것만 더 강화**하게 됩니다.
우리가 올려야 할 건 `maj@k`이고, 그건 **1~3/4 구간**에서 나옵니다. 진단에서 본 저확신 78문제가 정확히 여기예요.

**③ 잘린 풀이 제외** — `finish_reason`이 `stop`이 아니면 토큰 한도에 걸려 중간에 끊긴 것입니다. 답은 우연히 맞을 수 있어도 학습 데이터로는 나쁩니다.

**④ 중복 제거** — 같은 풀이가 여러 번 나오면 하나만.

In [ ]:
import json
from collections import Counter

records, stat = [], Counter()

for o, q, g in zip(outs, work["question"], work["answer"]):
    good = []
    for c in o.outputs:
        if c.finish_reason != "stop":        # 토큰 한도로 잘린 풀이 제외
            continue
        if parse_answer(c.text) == int(g):   # 정답만 채택
            good.append(c.text.strip())

    n_correct = len(good)
    stat[n_correct] += 1
    if n_correct == 0:
        continue

    # 중복 제거 (앞 300자 기준)
    seen, uniq = set(), []
    for s in good:
        key = " ".join(s[:300].split())
        if key not in seen:
            seen.add(key); uniq.append(s)

    # 난이도별 채택 개수: 4/4는 1개, 1~3/4는 최대 2개
    take = 1 if n_correct == K else 2
    for s in uniq[:take]:
        records.append({"question": q, "solution": s, "answer": int(g),
                        "n_correct": n_correct})

print("문제별 정답 개수 분포")
for k in range(K + 1):
    n = stat[k]
    print(f"  {k}/{K} 정답: {n:5d}문제 ({n/len(outs):5.1%})")

hard = sum(records[i]["n_correct"] < K for i in range(len(records)))
print(f"\n최종 학습 샘플: {len(records):,}개")
print(f"  그중 흔들리는 문제(1~{K-1}/{K}) 출신: {hard:,}개 ({hard/len(records):.1%})")
print(f"\n※ 0/{K} 문제 = 모델이 못 푸는 문제. pass@k 천장이 여기서 보입니다.")

---
## [9] 저장 + 확인 ▶️

**JSONL** 형식으로 저장합니다. 한 줄에 JSON 하나씩 들어가는 형식으로, 대용량 학습 데이터의 표준입니다.
(CSV와 달리 줄바꿈·따옴표가 많은 긴 텍스트를 안전하게 담을 수 있습니다)

### 저장 후 반드시 할 일
`/kaggle/working/train_rft.jsonl`은 **세션을 끄면 사라집니다.**
→ **Save Version → Save & Run All**로 Output에 남기거나, 직접 다운로드하세요.
→ 다음 노트북(학습)에서 이걸 **Kaggle Dataset으로 추가**해서 씁니다.

In [ ]:
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

import os
print(f"저장 완료: {OUT_PATH}  ({os.path.getsize(OUT_PATH)/1e6:.1f} MB, {len(records):,}줄)")

print("\n" + "="*70)
print("샘플 1개 (흔들리는 문제 출신)")
print("="*70)
ex = next((r for r in records if r["n_correct"] < K), records[0])
print("[문제]", ex["question"][:200])
print("\n[풀이]", ex["solution"][:600])
print("\n[정답]", ex["answer"], "| 4회 중", ex["n_correct"], "회 정답")

from IPython.display import FileLink
FileLink(OUT_PATH.split("/")[-1])